In [3]:
%pip install langchain openai python-dotenv
%pip install langsmith

import os
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable # Need to enable tracing on LangSmith
# --- Setup ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

print("API Key:", os.getenv("LANGCHAIN_API_KEY")[:6])
import os
from dotenv import load_dotenv
from openai import OpenAI
from langsmith import traceable # Need to enable tracing on LangSmith
# --- Setup ---
load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")

print("API Key:", os.getenv("LANGCHAIN_API_KEY")[:6])
print("Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Tracing:", os.getenv("LANGCHAIN_TRACING_V2"))

# Initialize model
client = OpenAI(api_key=my_api_key)


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
API Key: lsv2_p
API Key: lsv2_p
Project: Voice_Assistant_Project
Tracing: true



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

# --- PROMPTS ---

SYSTEM_PROMPT_COMPARATIVE_EVAL = """
You are a Senior Linguistics Consultant and Professional Communication Coach. 
Analyze the BASELINE vs. the CURRENT transcript.

Evaluation Criteria:
1. Grammar Accuracy (25%)
2. Clarity and Coherence (25%)
3. Vocabulary Richness (25%)
4. Sentence Structure (25%)

Output ONLY a JSON object with:
{
  "delta_scores": {"grammar": "+/-num", "clarity": "+/-num", "vocabulary": "+/-num", "structure": "+/-num"},
  "comparison_summary": "1-2 lines on progress",
  "wins": ["specific improvement 1", "specific improvement 2"],
  "focus_areas": ["target 1", "target 2"]
}
"""

SYSTEM_PROMPT_FILLER_WORDS = """
Identify filler words (um, uh, like, you know) and errors. 
Wrap ONLY those words in *asterisks*. Return the full text with highlights.
"""


In [5]:

# --- AGENT NODES ---

@traceable(run_type="tool")
def transcribe_audio(audio_file_path: str) -> str:
    """Step 1: Convert Speech to Text"""
    print(f"🎙️ Transcribing: {audio_file_path}")
    with open(audio_file_path, "rb") as audio_file:
        # Using whisper-1 as the standard reliable production model
        transcript = client.audio.transcriptions.create(
            file=audio_file,
            model="whisper-1"
        )
    return transcript.text

@traceable(run_type="llm")
def highlight_fillers(text: str) -> str:
    """Step 2: Visualize verbal tics and fillers"""
    print("✨ Highlighting filler words...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.1,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_FILLER_WORDS},
            {"role": "user", "content": text}
        ]
    )
    return response.choices[0].message.content.strip()

@traceable(run_type="llm")
def compare_progress(previous_text: str, current_text: str) -> dict:
    """Step 3: Comparative Analysis against Baseline"""
    print("Performing Comparative Evaluation...")
    comparison_query = f"BASELINE: {previous_text}\n\nCURRENT: {current_text}"
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        response_format={ "type": "json_object" },
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_COMPARATIVE_EVAL},
            {"role": "user", "content": comparison_query}
        ]
    )
    return json.loads(response.choices[0].message.content)


In [8]:

# --- MAIN EXECUTION ---

if __name__ == "__main__":
    # Settings
    AUDIO_FILE = "ElevenLabs_2025-11-22T03_21_45_Rachel_ Subject-Verb Agreement.mp3"
    
    # 1. Simulate a previous "Baseline" record from a database
    baseline_transcript = "I think the meeting went okay but we talked about stuff and it was like fine."

    try:
        # A. Get New Transcript
        current_text = transcribe_audio(AUDIO_FILE)
        
        # B. Analyze for verbal tics
        highlighted = highlight_fillers(current_text)
        
        # C. Compare with Baseline
        report = compare_progress(baseline_transcript, current_text)

        # --- FINAL OUTPUT ---
        print("\n" + "="*40)
        print("SPEECH INTELLIGENCE REPORT")
        print("="*40)
        print(f"\n[TRANSCRIPT HIGHLIGHTS]:\n{highlighted}")
        print(f"\n[PROGRESS SUMMARY]:\n{report['comparison_summary']}")
        
        print("\n[WINS]:")
        for win in report['wins']: print(f" ✅ {win}")
            
        print("\n[SCORE DELTAS]:")
        for cat, score in report['delta_scores'].items():
            print(f"{cat.capitalize()}: {score}")

    except Exception as e:
        print(f"Error in Pipeline: {e}")

🎙️ Transcribing: ElevenLabs_2025-11-22T03_21_45_Rachel_ Subject-Verb Agreement.mp3
✨ Highlighting filler words...
Performing Comparative Evaluation...

SPEECH INTELLIGENCE REPORT

[TRANSCRIPT HIGHLIGHTS]:
Every morning he *go* to school with his friends. They *walks* together down the street and *talks* about their homework. The teacher always *say* that punctuality *are* important. My sister also *go* to the same school and she *love* her classes. The students in the classroom *enjoys* learning new things every day.

[PROGRESS SUMMARY]:
The current transcript shows a decline in grammatical accuracy and coherence, while vocabulary and structure remain basic.

[WINS]:

[SCORE DELTAS]:
Grammar: -10
Clarity: -5
Vocabulary: -5
Structure: -5
